In [ ]:
import os
os.system("pip install ultralytics -q")
os.system("git clone https://github.com/abdur75648/End-To-End-Urdu-OCR-WebApp.git")


# Find correct path
for root, dirs, files in os.walk("/kaggle/working"):
    for f in files:
        if f == "model.py":
            print(os.path.join(root, f))

In [ ]:
!bash /kaggle/working/End-To-End-Urdu-OCR-WebApp/download_files.sh


In [ ]:
import os, sys, torch
from PIL import Image

sys.path.append("/kaggle/working/End-To-End-Urdu-OCR-WebApp")
from read import text_recognizer
from model import Model
from utils import CTCLabelConverter
from ultralytics import YOLO

In [ ]:
# Supported file types
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

#Model Paths
GLYPHS_PATH    = "/kaggle/working/End-To-End-Urdu-OCR-WebApp/UrduGlyphs.txt"
RECOGNIZER_PTH = "/kaggle/working/best_norm_ED.pth"
DETECTOR_PT    = "/kaggle/working/yolov8m_UrduDoc.pt"

In [ ]:
def load_models(device):
    with open(GLYPHS_PATH, "r", encoding="utf-8") as f:
        content = "".join(line.strip("\n") for line in f.readlines()) + " "
    converter = CTCLabelConverter(content)
    recognition_model = Model(num_class=len(converter.character), device=device)
    recognition_model = recognition_model.to(device)
    recognition_model.load_state_dict(torch.load(RECOGNIZER_PTH, map_location=device))
    recognition_model.eval()
    detection_model = YOLO(DETECTOR_PT)
    return detection_model, recognition_model, converter

In [ ]:
def ocr_image(image_path, detection_model, recognition_model, converter, device):
    input_image = Image.open(image_path).convert("RGB")
    results = detection_model.predict(
        source=input_image, conf=0.2, imgsz=1280,
        save=False, nms=True, device=device
    )
    bounding_boxes = results[0].boxes.xyxy.cpu().numpy().tolist()
    bounding_boxes.sort(key=lambda x: x[1])   # top-to-bottom reading order
    texts = []
    for box in bounding_boxes:
        cropped = input_image.crop(box)
        texts.append(text_recognizer(cropped, recognition_model, converter, device))
    return "\n".join(texts)

In [ ]:
# Resume Functioanlity

def get_progress_file(output_txt):
    base, _ = os.path.splitext(output_txt)
    return base + ".progress"

def load_progress(progress_file):
    if not os.path.exists(progress_file):
        return set()
    with open(progress_file, "r", encoding="utf-8") as f:
        return {line.strip() for line in f if line.strip()}

def save_progress(progress_file, filename):
    with open(progress_file, "a", encoding="utf-8") as f:
        f.write(filename + "\n")

In [ ]:
import re

def natural_sort_key(s):
    """Split string into list of text and number chunks for natural sorting."""
    return [int(text) if text.isdigit() else text.lower()
            for text in re.split(r'(\d+)', s)]

In [ ]:
def process_directory(image_dir, output_dir="/kaggle/working"):
    image_dir     = os.path.abspath(image_dir)
    dir_name      = os.path.basename(image_dir.rstrip(os.sep))
    os.makedirs(output_dir, exist_ok=True)
    output_txt    = os.path.join(output_dir, dir_name + ".txt")
    progress_file = os.path.join(output_dir, dir_name + ".progress")

    all_images = sorted(
        (f for f in os.listdir(image_dir)
         if os.path.splitext(f)[1].lower() in IMAGE_EXTENSIONS),
        key=natural_sort_key
    )
    if not all_images:
        print(f"No supported images found in: {image_dir}")
        return

    done      = load_progress(progress_file)
    remaining = [f for f in all_images if f not in done]

    print(f"Directory  : {image_dir}")
    print(f"Output file: {output_txt}")
    print(f"Total pages: {len(all_images)}  |  Done: {len(done)}  |  Remaining: {len(remaining)}")

    if not remaining:
        print("All pages already processed. Nothing to do.")
        return

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    print("Loading models ...")
    detection_model, recognition_model, converter = load_models(device)
    print("Models loaded.\n")

    with open(output_txt, "a", encoding="utf-8") as out_f:
        for idx, filename in enumerate(remaining, start=1):
            image_path = os.path.join(image_dir, filename)
            print(f"[{idx}/{len(remaining)}] Processing: {filename}")
            try:
                text = ocr_image(image_path, detection_model, recognition_model, converter, device)
                out_f.write(f"### {filename} ###\n")
                out_f.write(text + "\n\n")
                out_f.flush()
                save_progress(progress_file, filename)
                print(f"           Done  ({len(text.splitlines())} lines recognised)")
            except Exception as e:
                print(f"           ERROR: {e}  -- skipping this page")

    print(f"\nFinished! Text saved to: {output_txt}")

In [ ]:
for i in range(1,19):
    process_directory(f"/kaggle/input/datasets/ishahzaibkhan/gugtugu-images/Guftagu_{i}")